# Field Scan : modèles maladies sans le Pérou, évalués sur le Pérou

But : savoir comment chaque réseau se comporte **dans un pays qu'il n'a jamais vu**, sans toucher au test du Bangladesh.
On prend les modèles entraînés **sans le Pérou** (`POIDS_PEROU = 0` dans `maladies_v2_kaggle.ipynb`) et on les fait répondre sur les feuilles saines et Sigatoka du Pérou, avec **les seuils réglés sur la validation tanzanienne** (enregistrés dans `resultats.json`). Rien n'est réentraîné ni réglé ici.

**Avant de lancer :** Settings → Accelerator → **GPU T4**. Dans Input : le jeu `feuilles-v2`, et les **sorties** des versions du notebook `maladies_v2_kaggle` qui ont tourné sans le Pérou (Add Input → Your Work → Notebooks → choisir la version).

In [ ]:
# 1. Données et modèles disponibles
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import json
from pathlib import Path
import numpy as np, pandas as pd, torch

def chercher(nom):
    # cherche un fichier dans tous les Inputs (en suivant aussi les liens)
    return sorted(Path(d) / nom for d, _, fichiers in os.walk('/kaggle/input', followlinks=True) if nom in fichiers)

RACINE = chercher('images.csv')[0].parent
essais = []
for chemin in chercher('resultats.json'):
    r = json.load(open(chemin))
    poids = sorted(chemin.parent.glob('maladies_v2_*.pt'))
    print(f"{r['reseau']:20s} Pérou {r['poids_perou']}   {chemin.parent}   poids trouvés : {len(poids)}")
    if r['poids_perou'] == 0 and poids:
        essais.append((r, poids[0], chemin.parent))
print('\nModèles évalués (entraînés sans le Pérou) :', [r['reseau'] for r, _, _ in essais])
assert essais, "aucun modèle entraîné sans le Pérou : ajouter leurs sorties dans Input"
appareil = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# 2. Photos du Pérou (saines et Sigatoka ; la Cordana n'a jamais été préparée)
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T, models

CLASSES = ['saine', 'black_sigatoka', 'fusarium']
images = pd.read_csv(RACINE / 'images.csv')
perou = images[images.jeu == 'perou'].reset_index(drop=True)
print(perou.classe.value_counts().to_dict())

transformation = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
class Feuilles(Dataset):
    def __init__(self, df):
        self.chemins, self.etiquettes = list(df.image), [CLASSES.index(c) for c in df.classe]
    def __len__(self):
        return len(self.chemins)
    def __getitem__(self, i):
        return transformation(Image.open(RACINE / self.chemins[i]).convert('RGB')), self.etiquettes[i]
charge = DataLoader(Feuilles(perou), 64, num_workers=4)

In [ ]:
# 3. Même réseau, mêmes 4 vues et même décision que dans maladies_v2_kaggle.ipynb
def construire(reseau):
    if reseau == 'mobilenet_v3_large':
        m = models.mobilenet_v3_large(weights=None)
        m.classifier[3] = torch.nn.Linear(m.classifier[3].in_features, 3)
        resume = lambda x: m.classifier[:3](torch.flatten(m.avgpool(m.features(x)), 1)); tete = lambda f: m.classifier[3](f)
    elif reseau.startswith('efficientnet_b'):
        m = getattr(models, reseau)(weights=None)
        m.classifier[1] = torch.nn.Linear(m.classifier[1].in_features, 3)
        resume = lambda x: torch.flatten(m.avgpool(m.features(x)), 1); tete = lambda f: m.classifier(f)
    elif reseau == 'convnext_tiny':
        m = models.convnext_tiny(weights=None)
        m.classifier[2] = torch.nn.Linear(m.classifier[2].in_features, 3)
        resume = lambda x: m.classifier[:2](m.avgpool(m.features(x))); tete = lambda f: m.classifier[2](f)
    return m.to(appareil), resume, tete

VUES = [lambda x: x, lambda x: x.flip(3), lambda x: x.flip(2), lambda x: x.rot90(1, (2, 3))]

def predire(m, resume, tete, quatre_vues):
    m.eval(); probas, resumes = [], []
    vues = VUES if quatre_vues else VUES[:1]
    with torch.no_grad():
        for x, _ in charge:
            x = x.to(appareil); p, f = 0, 0
            for vue in vues:
                fv = resume(vue(x)); p = p + torch.softmax(tete(fv), 1); f = f + fv
            probas.append((p / len(vues)).cpu()); resumes.append((f / len(vues)).cpu())
    return torch.cat(probas).numpy(), torch.cat(resumes).numpy()

def decider(probas, resumes, r, dossier):
    p_malade = probas[:, 1:].sum(1)
    part = probas[:, 1:].max(1) / np.maximum(p_malade, 1e-9)
    nom = np.array(CLASSES)[1 + probas[:, 1:].argmax(1)]
    reponse = np.where(p_malade < r['seuil_alerte'], 'saine',
                       np.where(part >= r['seuil_nom'], nom, 'malade_non_identifiee')).astype(object)
    inconnu = probas.max(1) < r['seuil_confiance']
    if np.isfinite(r['seuil_distance']):
        stats = np.load(dossier / 'rejet_inconnu.npz')
        d = np.min([np.einsum('ij,jk,ik->i', resumes - c, stats['precision_cov'], resumes - c) for c in stats['centres']], 0)
        inconnu |= d > r['seuil_distance']
    reponse[inconnu] = 'inconnu'
    return reponse

In [ ]:
# 4. Résultats sur le Pérou, pays jamais vu par ces modèles
MALADE = ['black_sigatoka', 'fusarium', 'malade_non_identifiee']
saine, sig = (perou.classe == 'saine').values, (perou.classe == 'black_sigatoka').values
lignes = []
for r, poids, dossier in essais:
    m, resume, tete = construire(r['reseau'])
    m.load_state_dict(torch.load(poids, map_location=appareil))
    probas, resumes = predire(m, resume, tete, r.get('quatre_vues', True))
    rep = decider(probas, resumes, r, dossier)
    regle_05 = np.where(probas[:, 1:].sum(1) >= 0.5, 'malade', 'saine')
    lignes.append({
        'reseau': r['reseau'],
        'saines_bien_reconnues': round((rep[saine] == 'saine').mean(), 3),
        'fausses_alertes_saines': round(np.isin(rep[saine], MALADE).mean(), 3),
        'saines_inconnu': round((rep[saine] == 'inconnu').mean(), 3),
        'sigatoka_signalee_malade': round(np.isin(rep[sig], MALADE).mean(), 3),
        'sigatoka_bien_nommee': round((rep[sig] == 'black_sigatoka').mean(), 3),
        'sigatoka_dite_saine': round((rep[sig] == 'saine').mean(), 3),
        'sigatoka_inconnu': round((rep[sig] == 'inconnu').mean(), 3),
        'regle_05_fausses_alertes_saines': round((regle_05[saine] == 'malade').mean(), 3),
        'regle_05_sigatoka_dite_saine': round((regle_05[sig] == 'saine').mean(), 3),
    })
    del m; torch.cuda.empty_cache()
resultats = pd.DataFrame(lignes).set_index('reseau')
print(f"Pérou : {saine.sum()} saines, {sig.sum()} Sigatoka\n")
print(resultats.T.to_string())
resultats.to_csv('/kaggle/working/evaluation_perou.csv')